# Fase 2 — Claude Sonnet 5.5 / Haiku 5.5 · SOLO OFFLINE

Questo notebook NON esegue alcuna inferenza Anthropic e NON modifica i dati V3 o Gemini 5+1. Usa una directory Colab separata da `/content/Tesi_OpenRCA`.

Prima dei test API: verifica la versione codice, le dipendenze, la suite Python e la disponibilità del Secret `ANTHROPIC_API_KEY` (non lo stampa). **Non modificare la vecchia sessione Colab congelata per Gemini.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

CLAUDE_REPO = Path('/content/Tesi_OpenRCA_Claude')
CLAUDE_DATA = Path('/content/drive/MyDrive/Tesi_OpenRCA')
BRANCH = 'feature/claude-cross-model-2026-11-05'
URL = 'https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git'

if not CLAUDE_REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,
                    URL, str(CLAUDE_REPO)], check=True)

def claude_git(*args):
    return subprocess.run(['git', '-C', str(CLAUDE_REPO), *args],
                          check=True, text=True,
                          capture_output=True).stdout.strip()

assert claude_git('branch', '--show-current') == BRANCH
assert not claude_git('status', '--porcelain')
print('Branch Claude:', BRANCH)
print('Commit Claude:', claude_git('rev-parse', 'HEAD'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(CLAUDE_REPO / 'requirements.txt')], check=True)
try:
    print('LangChain Anthropic:', version('langchain-anthropic'))
except PackageNotFoundError as exc:
    raise RuntimeError('Manca langchain-anthropic') from exc

for rel in ['data/processed/split_market_v1.csv',
            'data/processed/benchmark_inputs_market.csv', 'Market.zip']:
    p = CLAUDE_DATA / rel
    print(rel, 'PRESENTE' if p.exists() else 'ASSENTE')

print('Bootstrap Claude OFFLINE completato: nessuna API chiamata.')

## Test di regressione senza chiave API

Prima si eseguono i test del provider Claude, poi la suite generale. Nessuna inferenza a pagamento: per i test del provider si usano mock.

In [ ]:
env = {**os.environ, 'PYTHONPATH': str(CLAUDE_REPO / 'src')}
for mask in ['test_anthropic_model_factory.py', 'test_*.py']:
    print('TEST:', mask)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                    '-s', str(CLAUDE_REPO / 'src/tests'),
                    '-p', mask, '-v'],
                   cwd=CLAUDE_REPO, env=env, check=True)
print('Suite terminata. Nessuna API Claude chiamata.')

## Verifica del Secret — nessun costo

Apri Colab → pannello **Secrets** (icona chiave) → aggiungi **ANTHROPIC_API_KEY** → incolla il valore consegnato dal professore → abilita l'accesso al notebook. Non inserirlo qui come testo e non salvarlo su Drive o GitHub.

La seguente cella legge solo il Secret. Una chiave presente non conferma che sia valida o che l'account disponga di credito; quello richiede uno smoke test successivo e autorizzato.

In [ ]:
from google.colab import userdata
secret = userdata.get('ANTHROPIC_API_KEY')
assert isinstance(secret, str) and secret.strip(), (
    'Secret ANTHROPIC_API_KEY assente o non accessibile.'
)
os.environ['ANTHROPIC_API_KEY'] = secret
del secret

print('ANTHROPIC_API_KEY accessibile: SI')
print('Modelli in piano: claude-sonnet-5-5, claude-haiku-5-5')
print('Nessuna inferenza, nessuna chiamata API, nessun file modificato.')
